# xA Model — StatsBomb Open Data

## Architettura

```
xA(pass) = P_shot(pass) × xG_model(shot)
```

- **P_shot**: classificatore binario su tutti i passaggi completati — P(pass → tiro)
- **xG_model**: modello xG già allenato (`xg_model_360_no_penalty.joblib`) — applicato al tiro risultante

Ogni passaggio completato riceve un valore xA:
- Key pass: `P_shot × xG_model(shot_features)`
- Non-key pass: `P_shot × zone_avg_xg(end_location)` — xG medio dei tiri storici da quella zona

**Struttura**:
1. Config & coverage
2. EDA
3. Feature engineering (passaggi)
4. Dataset A — tutti i passaggi completati
5. Split train/val
6. Model 1 — P_shot
7. Caricamento xG model + shot features
8. Zone xG lookup — fallback per non-key pass
9. xA = P_shot × xG
10. Feature importance
11. Salvataggio


# 1. Config & Setup

In [ ]:
from pathlib import Path
import json

DATA_ROOT      = Path("/Users/lorenzoguercio/Documents/Projects/sport_data/open-data/data")
EVENTS_DIR     = DATA_ROOT / "events"
THREESIXTY_DIR = DATA_ROOT / "three-sixty"
PROJECT_ROOT   = Path("/Users/lorenzoguercio/Documents/Projects/sport_code_repo")
MODELS_DIR     = PROJECT_ROOT / "xgoals_project" / "models"

HOLDOUT_MATCH_IDS   = ["3788741", "3788743"]
INCOMPLETE_OUTCOMES = {'Incomplete', 'Out', 'Pass Offside', 'Unknown'}


# 2. Data Coverage

In [ ]:
event_ids        = {p.stem for p in EVENTS_DIR.glob('*.json')}
three_sixty_ids  = {p.stem for p in THREESIXTY_DIR.glob('*.json')}
matches_with_360 = sorted(event_ids & three_sixty_ids)

train_match_ids   = [m for m in matches_with_360 if m not in HOLDOUT_MATCH_IDS]
holdout_match_ids = [m for m in matches_with_360 if m in HOLDOUT_MATCH_IDS]

print(f"Matches con 360:  {len(matches_with_360)} / {len(event_ids)}")
print(f"Train: {len(train_match_ids)}  |  Holdout: {len(holdout_match_ids)}")


# 3. EDA

## 3.1 Shot assist rate

In [ ]:
SAMPLE = train_match_ids[:20]
total_passes = total_shot_assist = total_goal_assist = 0

for mid in SAMPLE:
    ev = json.loads((EVENTS_DIR / f"{mid}.json").read_text())
    for e in ev:
        if e.get('type', {}).get('name') != 'Pass':
            continue
        p = e.get('pass', {})
        outcome_name = p.get('outcome', {}).get('name') if p.get('outcome') else None
        if outcome_name in INCOMPLETE_OUTCOMES:
            continue
        total_passes += 1
        if p.get('shot_assist'):  total_shot_assist += 1
        if p.get('goal_assist'):  total_goal_assist += 1

print(f"Passaggi completati ({len(SAMPLE)} match): {total_passes:,}")
print(f"Shot assist: {total_shot_assist} ({total_shot_assist/total_passes*100:.2f}%)")
print(f"Goal assist: {total_goal_assist} ({total_goal_assist/total_passes*100:.2f}%)")


## 3.2 Shot assist rate per tipo e play pattern

In [ ]:
import matplotlib.pyplot as plt

height_shot = {};  height_total = {}
pattern_shot = {}; pattern_total = {}
cross_shot = cross_total = 0

for mid in train_match_ids:
    ev = json.loads((EVENTS_DIR / f"{mid}.json").read_text())
    for e in ev:
        if e.get('type', {}).get('name') != 'Pass':
            continue
        p = e.get('pass', {})
        outcome_name = p.get('outcome', {}).get('name') if p.get('outcome') else None
        if outcome_name in INCOMPLETE_OUTCOMES:
            continue
        h  = p.get('height', {}).get('name', 'Unknown')
        pp = e.get('play_pattern', {}).get('name', 'Unknown')
        is_sa = bool(p.get('shot_assist'))
        height_total[h]   = height_total.get(h, 0) + 1
        height_shot[h]    = height_shot.get(h, 0) + int(is_sa)
        pattern_total[pp] = pattern_total.get(pp, 0) + 1
        pattern_shot[pp]  = pattern_shot.get(pp, 0) + int(is_sa)
        if p.get('cross'):
            cross_total += 1
            cross_shot  += int(is_sa)

heights  = sorted(height_total)
h_rate   = [height_shot.get(h, 0) / height_total[h] * 100 for h in heights]
patterns = sorted(pattern_total, key=lambda x: -pattern_total[x])[:6]
p_rate   = [pattern_shot.get(p, 0) / pattern_total[p] * 100 for p in patterns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(heights, h_rate, color='steelblue')
axes[0].set_ylabel('Shot assist rate (%)')
axes[0].set_title('Per altezza del passaggio')
axes[0].tick_params(axis='x', rotation=15)
axes[1].barh(patterns, p_rate, color='darkorange')
axes[1].set_xlabel('Shot assist rate (%)')
axes[1].set_title('Per play pattern')
plt.tight_layout(); plt.show()
print(f"Cross shot assist rate: {cross_shot}/{cross_total} = {cross_shot/cross_total*100:.1f}%")


# 4. Feature Engineering — Passaggi

Feature usate da **P_shot** (su tutti i passaggi):
- **Geometria**: posizione ricevitore, distanza/angolo alla porta, vettore del pass
- **Metadata**: altezza, arto, cross, switch, through_ball, tecnica
- **360**: pressione sul ricevitore, difensori tra ricevitore e porta

In [ ]:
import math

GOAL_X     = 120
GOAL_Y     = 40
GOAL_WIDTH = 7.32


def geometry_features(start_loc, end_loc):
    if not (isinstance(start_loc, list) and isinstance(end_loc, list)):
        return {}
    sx, sy = start_loc[0], start_loc[1]
    ex, ey = end_loc[0], end_loc[1]
    if sx < 60:
        sx, sy = 120 - sx, 80 - sy
        ex, ey = 120 - ex, 80 - ey
    recv_dist  = math.hypot(ex - GOAL_X, ey - GOAL_Y)
    angle_den  = (ex - GOAL_X)**2 + (ey - GOAL_Y)**2 - (GOAL_WIDTH / 2)**2
    recv_angle = max(0.0, math.atan2(GOAL_WIDTH * abs(ey - GOAL_Y), angle_den)) if angle_den != 0 else 0.0
    return {
        'start_x': sx, 'start_y': sy, 'end_x': ex, 'end_y': ey,
        'pass_length':         math.hypot(ex - sx, ey - sy),
        'pass_forward':        ex - sx,
        'receiver_dist_goal':  recv_dist,
        'receiver_angle_goal': recv_angle,
        'passer_dist_goal':    math.hypot(sx - GOAL_X, sy - GOAL_Y),
        'cross_field_y':       abs(ey - GOAL_Y),
    }


def pass_meta_features(pass_data):
    return {
        'height':       pass_data.get('height', {}).get('name'),
        'body_part':    pass_data.get('body_part', {}).get('name'),
        'technique':    pass_data.get('technique', {}).get('name'),
        'cross':        bool(pass_data.get('cross')),
        'switch':       bool(pass_data.get('switch')),
        'through_ball': bool(pass_data.get('through_ball')),
    }


def extract_360_pass_features(frame, end_loc):
    ff = frame.get('freeze_frame') or []
    if not isinstance(end_loc, list):
        return {}
    ex, ey = end_loc[0], end_loc[1]
    opponents = [p for p in ff if p.get('teammate') is False and not p.get('keeper')]
    opp_locs  = [p['location'] for p in opponents if isinstance(p.get('location'), list)]
    if opp_locs:
        dists       = [math.hypot(l[0]-ex, l[1]-ey) for l in opp_locs]
        nearest_def = min(dists)
        n_def_2m = sum(1 for d in dists if d <= 2)
        n_def_3m = sum(1 for d in dists if d <= 3)
        n_def_5m = sum(1 for d in dists if d <= 5)
    else:
        nearest_def = float('nan')
        n_def_2m = n_def_3m = n_def_5m = 0
    n_def_recv_goal = sum(
        1 for l in opp_locs if l[0] > ex and abs(l[1] - GOAL_Y) <= (GOAL_WIDTH/2 + 3)
    )
    teammates = [p for p in ff if p.get('teammate') is True and not p.get('actor')]
    tm_locs   = [p['location'] for p in teammates if isinstance(p.get('location'), list)]
    n_tm_5m   = sum(1 for l in tm_locs if math.hypot(l[0]-ex, l[1]-ey) <= 5)
    return {
        'nearest_def_to_receiver':  nearest_def,
        'n_def_2m_receiver':        n_def_2m,
        'n_def_3m_receiver':        n_def_3m,
        'n_def_5m_receiver':        n_def_5m,
        'n_def_between_recv_goal':  n_def_recv_goal,
        'n_teammates_5m_receiver':  n_tm_5m,
    }


# 5. Dataset A — Tutti i passaggi completati

Target: `shot_assist` (1/0) per il modello P_shot.

In [ ]:
import pandas as pd

rows_a = []

for mid in train_match_ids:
    ev = json.loads((EVENTS_DIR / f"{mid}.json").read_text())
    try:
        th = json.loads((THREESIXTY_DIR / f"{mid}.json").read_text())
        frames_idx = {f['event_uuid']: f for f in th}
    except Exception:
        frames_idx = {}

    for e in ev:
        if e.get('type', {}).get('name') != 'Pass':
            continue
        p = e.get('pass', {})
        outcome_name = p.get('outcome', {}).get('name') if p.get('outcome') else None
        if outcome_name in INCOMPLETE_OUTCOMES:
            continue

        event_id   = e.get('id')
        start_loc  = e.get('location')
        end_loc    = p.get('end_location')
        frame      = frames_idx.get(event_id, {})

        rows_a.append({
            'match_id':    mid,
            'event_id':    event_id,
            'shot_assist': int(bool(p.get('shot_assist'))),
            'play_pattern': e.get('play_pattern', {}).get('name'),
            'has_360':     bool(frame),
            **geometry_features(start_loc, end_loc),
            **pass_meta_features(p),
            **(extract_360_pass_features(frame, end_loc) if frame else {}),
        })

df_all = pd.DataFrame(rows_a)
print(f"Dataset A: {len(df_all):,} passaggi")
print(f"shot_assist: {df_all['shot_assist'].sum():,} ({df_all['shot_assist'].mean()*100:.2f}%)")
print(f"Con 360: {df_all['has_360'].sum():,} ({df_all['has_360'].mean()*100:.1f}%)")


# 6. Train / Val Split (livello match)

In [ ]:
from sklearn.model_selection import train_test_split

all_match_ids          = df_all['match_id'].unique()
train_ids, val_ids     = train_test_split(all_match_ids, test_size=0.2, random_state=42)
train_ids_set          = set(train_ids)
val_ids_set            = set(val_ids)

train_a = df_all[df_all['match_id'].isin(train_ids_set)].copy()
val_a   = df_all[df_all['match_id'].isin(val_ids_set)].copy()

print(f"Train: {len(train_a):,} | shot_assist rate {train_a['shot_assist'].mean()*100:.2f}%")
print(f"Val:   {len(val_a):,}   | shot_assist rate {val_a['shot_assist'].mean()*100:.2f}%")


# 7. Feature Set

In [ ]:
BASE_NUM = [
    'receiver_dist_goal', 'receiver_angle_goal', 'passer_dist_goal',
    'pass_length', 'pass_forward', 'cross_field_y',
]
BASE_CAT = ['height', 'body_part', 'play_pattern', 'technique']
BASE_BIN = ['cross', 'switch', 'through_ball']
FEAT_360 = [
    'nearest_def_to_receiver', 'n_def_2m_receiver', 'n_def_3m_receiver',
    'n_def_5m_receiver', 'n_def_between_recv_goal', 'n_teammates_5m_receiver',
]
ALL_COLS = BASE_NUM + BASE_BIN + BASE_CAT + FEAT_360


# 8. Model 1 — P_shot

Classificatore binario su tutti i passaggi completati.
Target: `shot_assist`. Confronto baseline vs 360-enhanced.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score


def build_classifier(num_cols, cat_cols, bin_cols):
    prep = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), cat_cols),
        ('bin', SimpleImputer(strategy='constant', fill_value=0), bin_cols),
    ], remainder='drop')
    return Pipeline([
        ('prep', prep),
        ('clf', LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            min_child_samples=20, class_weight='balanced',
            random_state=42, verbose=-1,
        )),
    ])


def eval_clf(model, X, y, label=''):
    proba = model.predict_proba(X)[:, 1]
    print(f"{label:25s}  log_loss={log_loss(y, proba):.4f}  "
          f"brier={brier_score_loss(y, proba):.4f}  AUC={roc_auc_score(y, proba):.4f}")
    return proba


X_train_a = train_a[ALL_COLS]
X_val_a   = val_a[ALL_COLS]
y_train_a = train_a['shot_assist']
y_val_a   = val_a['shot_assist']

pshot_base = build_classifier(BASE_NUM, BASE_CAT, BASE_BIN)
pshot_base.fit(X_train_a, y_train_a)

pshot_360 = build_classifier(BASE_NUM + FEAT_360, BASE_CAT, BASE_BIN)
pshot_360.fit(X_train_a, y_train_a)

print("=== Model P_shot — Validation ===")
proba_pshot_base = eval_clf(pshot_base, X_val_a, y_val_a, 'Baseline')
proba_pshot_360  = eval_clf(pshot_360,  X_val_a, y_val_a, '360-enhanced')


# 9. xG Model — Caricamento e Feature dei Tiri

Il modello xG già allenato (`xg_model_360_no_penalty.joblib`) viene applicato ai tiri
risultanti dai key pass. Le feature attese dal modello sono:

```
num: distance, angle, nearest_defender_dist, keeper_dist_to_shot,
     keeper_dist_to_goal, keeper_present, n_defenders_within_1m/2m/3m,
     n_players_in_cone_to_goal, visible_area_size,
     first_time, one_on_one, under_pressure
cat: body_part, shot_type, shot_technique, play_pattern
```

In [ ]:
import joblib
import numpy as np

xg_model = joblib.load(MODELS_DIR / 'xg_model_360_no_penalty.joblib')
print("xG model caricato:", xg_model)


## 9.1 Funzione di estrazione feature tiro (compatibile con xG model)

In [ ]:
def polygon_area(coords):
    if not coords or len(coords) < 3:
        return 0.0
    n = len(coords)
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += coords[i][0] * coords[j][1]
        area -= coords[j][0] * coords[i][1]
    return abs(area) / 2.0


def point_in_triangle(p, a, b, c):
    """Controlla se il punto p è dentro il triangolo abc (cross product method)."""
    def sign(p1, p2, p3):
        return (p1[0]-p3[0])*(p2[1]-p3[1]) - (p2[0]-p3[0])*(p1[1]-p3[1])
    d1, d2, d3 = sign(p, a, b), sign(p, b, c), sign(p, c, a)
    has_neg = (d1 < 0) or (d2 < 0) or (d3 < 0)
    has_pos = (d1 > 0) or (d2 > 0) or (d3 > 0)
    return not (has_neg and has_pos)


LEFT_POST  = (GOAL_X, GOAL_Y - GOAL_WIDTH / 2)
RIGHT_POST = (GOAL_X, GOAL_Y + GOAL_WIDTH / 2)


def extract_shot_features(shot_event, frame_360):
    """Estrae le feature del tiro nel formato atteso dall'xG model."""
    shot_data = shot_event.get('shot', {})
    loc = shot_event.get('location', [])
    if not isinstance(loc, list) or len(loc) < 2:
        return None

    x, y = loc[0], loc[1]
    if x < 60:
        x, y = 120 - x, 80 - y

    # Geometria
    distance  = math.hypot(x - GOAL_X, y - GOAL_Y)
    angle_den = (x - GOAL_X)**2 + (y - GOAL_Y)**2 - (GOAL_WIDTH / 2)**2
    angle     = max(0.0, math.atan2(GOAL_WIDTH * abs(y - GOAL_Y), angle_den)) if angle_den != 0 else 0.0

    # Metadata tiro
    first_time     = float(bool(shot_data.get('first_time')))
    one_on_one     = float(bool(shot_data.get('one_on_one')))
    under_pressure = float(bool(shot_event.get('under_pressure')))
    body_part      = shot_data.get('body_part', {}).get('name')
    shot_type      = shot_data.get('type', {}).get('name')
    shot_technique = shot_data.get('technique', {}).get('name')
    play_pattern   = shot_event.get('play_pattern', {}).get('name')

    # Feature 360 dal freeze_frame del tiro
    ff = frame_360.get('freeze_frame', []) if frame_360 else []
    shot_pt = (x, y)

    opponents = [p for p in ff if p.get('teammate') is False and not p.get('keeper')]
    opp_locs  = [p['location'] for p in opponents if isinstance(p.get('location'), list)]

    keeper_list = [p for p in ff if p.get('keeper') and not p.get('teammate')]
    keeper_loc  = keeper_list[0]['location'] if keeper_list and isinstance(keeper_list[0].get('location'), list) else None

    if opp_locs:
        dists = [math.hypot(l[0]-x, l[1]-y) for l in opp_locs]
        nearest_defender_dist = min(dists)
        n_def_1m = sum(1 for d in dists if d <= 1)
        n_def_2m = sum(1 for d in dists if d <= 2)
        n_def_3m = sum(1 for d in dists if d <= 3)
    else:
        nearest_defender_dist = float('nan')
        n_def_1m = n_def_2m = n_def_3m = 0

    keeper_present     = float(keeper_loc is not None)
    keeper_dist_to_shot = math.hypot(keeper_loc[0]-x, keeper_loc[1]-y) if keeper_loc else float('nan')
    keeper_dist_to_goal = math.hypot(keeper_loc[0]-GOAL_X, keeper_loc[1]-GOAL_Y) if keeper_loc else float('nan')

    all_locs = [p['location'] for p in ff
                if isinstance(p.get('location'), list) and not p.get('actor')]
    n_players_in_cone = sum(
        1 for l in all_locs
        if point_in_triangle(l, shot_pt, LEFT_POST, RIGHT_POST)
    )

    visible_area      = frame_360.get('visible_area', []) if frame_360 else []
    visible_area_size = polygon_area(visible_area)

    return {
        'distance':               distance,
        'angle':                  angle,
        'nearest_defender_dist':  nearest_defender_dist,
        'keeper_dist_to_shot':    keeper_dist_to_shot,
        'keeper_dist_to_goal':    keeper_dist_to_goal,
        'keeper_present':         keeper_present,
        'n_defenders_within_1m':  n_def_1m,
        'n_defenders_within_2m':  n_def_2m,
        'n_defenders_within_3m':  n_def_3m,
        'n_players_in_cone_to_goal': n_players_in_cone,
        'visible_area_size':      visible_area_size,
        'first_time':             first_time,
        'one_on_one':             one_on_one,
        'under_pressure':         under_pressure,
        'body_part':              body_part,
        'shot_type':              shot_type,
        'shot_technique':         shot_technique,
        'play_pattern':           play_pattern,
    }


## 9.2 Dataset dei tiri risultanti dai key pass + xG predictions

In [ ]:
# Costruisce shot_df: una riga per ogni key pass con le feature del tiro risultante
shot_rows = []

for mid in train_match_ids:
    ev = json.loads((EVENTS_DIR / f"{mid}.json").read_text())
    try:
        th = json.loads((THREESIXTY_DIR / f"{mid}.json").read_text())
        frames_idx = {f['event_uuid']: f for f in th}
    except Exception:
        frames_idx = {}

    # Index: key_pass_id -> shot event
    shots_idx = {
        e['shot'].get('key_pass_id'): e
        for e in ev
        if e.get('type', {}).get('name') == 'Shot' and e.get('shot', {}).get('key_pass_id')
    }

    for e in ev:
        if not e.get('pass', {}).get('shot_assist'):
            continue
        kp_id   = e.get('id')
        shot    = shots_idx.get(kp_id)
        if shot is None:
            continue

        shot_id   = shot.get('id')
        frame_360 = frames_idx.get(shot_id, {})
        feat      = extract_shot_features(shot, frame_360)
        if feat is None:
            continue

        shot_rows.append({
            'match_id':        mid,
            'kp_event_id':     kp_id,
            'shot_event_id':   shot_id,
            'is_goal':         int(shot.get('shot', {}).get('outcome', {}).get('name') == 'Goal'),
            'statsbomb_xg':    shot.get('shot', {}).get('statsbomb_xg'),
            **feat,
        })

shot_df = pd.DataFrame(shot_rows)

# Applica il modello xG per ottenere our_xg
XG_COLS = [
    'distance', 'angle', 'nearest_defender_dist', 'keeper_dist_to_shot',
    'keeper_dist_to_goal', 'keeper_present', 'n_defenders_within_1m',
    'n_defenders_within_2m', 'n_defenders_within_3m',
    'n_players_in_cone_to_goal', 'visible_area_size',
    'first_time', 'one_on_one', 'under_pressure',
    'body_part', 'shot_type', 'shot_technique', 'play_pattern',
]
shot_df['our_xg'] = xg_model.predict_proba(shot_df[XG_COLS])[:, 1]

print(f"Shot dataset: {len(shot_df):,} tiri da key pass")
print(f"Goal: {shot_df['is_goal'].sum()} ({shot_df['is_goal'].mean()*100:.1f}%)")
print(f"our_xg medio:       {shot_df['our_xg'].mean():.4f}")
print(f"statsbomb_xg medio: {shot_df['statsbomb_xg'].mean():.4f}")

# Correlazione tra our_xg e statsbomb_xg (sanity check)
print(f"Correlazione our_xg vs statsbomb_xg: {shot_df[['our_xg','statsbomb_xg']].corr().iloc[0,1]:.4f}")


# 10. Zone xG Lookup — Fallback per Non-Key Pass

Per i passaggi che non portano a un tiro, il secondo fattore dell'xA è lo `our_xg` medio
dei tiri storicamente effettuati dalla zona corrispondente alla `end_location` del passaggio.

Griglia 12×8 sul campo (coordinate normalizzate attacking half).

In [ ]:
# Costruisce la zone_xg lookup dai tiri di training
import numpy as np

N_X_BINS, N_Y_BINS = 12, 8
x_bins = np.linspace(60, 120, N_X_BINS + 1)  # solo attacking half
y_bins = np.linspace(0, 80, N_Y_BINS + 1)

# Raccoglie tutti i tiri dal training set con our_xg
all_shot_rows = []

for mid in train_match_ids:
    if mid not in train_ids_set:
        continue
    ev = json.loads((EVENTS_DIR / f"{mid}.json").read_text())
    try:
        th = json.loads((THREESIXTY_DIR / f"{mid}.json").read_text())
        fi = {f['event_uuid']: f for f in th}
    except Exception:
        fi = {}

    for e in ev:
        if e.get('type', {}).get('name') != 'Shot':
            continue
        feat = extract_shot_features(e, fi.get(e.get('id'), {}))
        if feat is None:
            continue
        all_shot_rows.append(feat)

all_shots_df = pd.DataFrame(all_shot_rows)
all_shots_df['our_xg'] = xg_model.predict_proba(all_shots_df[XG_COLS])[:, 1]

# Calcola xG medio per zona (x normalizzato: attacking half sempre)
all_shots_df['x_bin'] = pd.cut(all_shots_df['distance'].apply(
    lambda d: max(60, GOAL_X - d)), bins=x_bins, labels=False, include_lowest=True
)
all_shots_df['y_bin'] = pd.cut(all_shots_df['angle'].apply(
    lambda a: min(79.9, GOAL_Y + a * 10)), bins=y_bins, labels=False, include_lowest=True
)

# Lookup: (end_x_bin, end_y_bin) -> avg our_xg
zone_xg_map = (
    all_shots_df.groupby(['x_bin', 'y_bin'])['our_xg']
    .mean()
    .to_dict()
)
global_avg_xg = all_shots_df['our_xg'].mean()

print(f"Tiri totali per zone lookup: {len(all_shots_df):,}")
print(f"Zone con dati: {len(zone_xg_map)}")
print(f"xG medio globale (fallback): {global_avg_xg:.4f}")


def zone_xg_lookup(end_x, end_y):
    """xG medio storico dei tiri effettuati dalla zona end_x, end_y."""
    # Normalizza in attacking direction
    if end_x < 60:
        end_x, end_y = 120 - end_x, 80 - end_y
    xi = int(np.digitize(end_x, x_bins) - 1)
    yi = int(np.digitize(end_y, y_bins) - 1)
    xi = max(0, min(xi, N_X_BINS - 1))
    yi = max(0, min(yi, N_Y_BINS - 1))
    return zone_xg_map.get((xi, yi), global_avg_xg)


# 11. xA = P_shot × xG

```
xA(key_pass)     = P_shot(pass_features) × our_xg(shot_features)
xA(non_key_pass) = P_shot(pass_features) × zone_xg(end_location)
```

In [ ]:
# P_shot su tutti i passaggi di validazione
proba_pshot = pshot_360.predict_proba(X_val_a)[:, 1]

val_a = val_a.copy()
val_a['p_shot'] = proba_pshot

# xG dal lookup di zona (per tutti i passaggi)
val_a['zone_xg'] = val_a.apply(
    lambda r: zone_xg_lookup(r.get('end_x', 60), r.get('end_y', 40)), axis=1
)

# Merge con our_xg del tiro reale (per i key pass)
shot_df_val = shot_df[shot_df['match_id'].isin(val_ids_set)][['kp_event_id', 'our_xg', 'is_goal', 'statsbomb_xg']]
val_a = val_a.merge(shot_df_val.rename(columns={'kp_event_id': 'event_id'}),
                    on='event_id', how='left')

# xA finale:
# - key pass: P_shot × our_xg (modello xG reale)
# - non-key:  P_shot × zone_xg (fallback)
val_a['xg_used'] = val_a['our_xg'].fillna(val_a['zone_xg'])
val_a['xa']      = val_a['p_shot'] * val_a['xg_used']

print(f"xA medio (tutti i pass):  {val_a['xa'].mean():.5f}")
print(f"xA medio (key pass):      {val_a.loc[val_a['shot_assist']==1, 'xa'].mean():.4f}")
print(f"xA medio (non-key pass):  {val_a.loc[val_a['shot_assist']==0, 'xa'].mean():.5f}")

kp_val = val_a[val_a['shot_assist'] == 1].copy()
print(f"\nSu key pass val: xA medio = {kp_val['xa'].mean():.4f}")
print(f"our_xg medio dal tiro:    {kp_val['our_xg'].mean():.4f}")
print(f"statsbomb_xg medio:       {kp_val['statsbomb_xg'].mean():.4f}")


In [ ]:
# Calibrazione match-level: somma xA key pass vs goal assist reali
match_summary = kp_val.groupby('match_id').agg(
    xa_sum=('xa', 'sum'),
    goals=('is_goal', 'sum'),
).reset_index()

print(f"xA sum medio per match: {match_summary['xa_sum'].mean():.3f}")
print(f"Goal medi per match:    {match_summary['goals'].mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(val_a['xa'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('xA'); axes[0].set_ylabel('Passaggi')
axes[0].set_title('Distribuzione xA — tutti i passaggi (val)')

axes[1].scatter(match_summary['xa_sum'], match_summary['goals'], alpha=0.6)
lim = max(match_summary['xa_sum'].max(), match_summary['goals'].max()) + 0.2
axes[1].plot([0, lim], [0, lim], 'k--', label='Perfetta calibrazione')
axes[1].set_xlabel('xA sum (predicted)'); axes[1].set_ylabel('Goal reali')
axes[1].set_title('Calibrazione xA — livello match'); axes[1].legend()

plt.tight_layout(); plt.show()


# 12. Feature Importance — P_shot

In [ ]:
from sklearn.inspection import permutation_importance

imp_cols = BASE_NUM + FEAT_360 + BASE_BIN + BASE_CAT
perm = permutation_importance(
    pshot_360, X_val_a[imp_cols], y_val_a,
    n_repeats=8, random_state=42, scoring='roc_auc'
)

sorted_idx = perm.importances_mean.argsort()[::-1][:15]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    [imp_cols[i] for i in sorted_idx[::-1]],
    perm.importances_mean[sorted_idx[::-1]],
    xerr=perm.importances_std[sorted_idx[::-1]],
    color='steelblue',
)
ax.set_xlabel('AUC decrease (permutation importance)')
ax.set_title('Feature Importance — P_shot (360-enhanced)')
plt.tight_layout(); plt.show()


# 13. Salvataggio

In [ ]:
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)

PSHOT_PATH    = MODELS_DIR / 'xa_pshot_360.joblib'
ZONE_XG_PATH  = MODELS_DIR / 'xa_zone_xg_lookup.joblib'
HOLDOUT_PATH  = MODELS_DIR / 'xa_holdout_match_ids.json'

joblib.dump(pshot_360,   PSHOT_PATH)
joblib.dump({'zone_xg_map': zone_xg_map, 'global_avg_xg': global_avg_xg,
             'x_bins': x_bins.tolist(), 'y_bins': y_bins.tolist()}, ZONE_XG_PATH)
HOLDOUT_PATH.write_text(json.dumps(HOLDOUT_MATCH_IDS))

print(f"P_shot model:    {PSHOT_PATH}")
print(f"Zone xG lookup:  {ZONE_XG_PATH}")
print(f"xG model (già esistente): {MODELS_DIR / 'xg_model_360_no_penalty.joblib'}")
print()
print("=== Inferenza ===")
print("key pass:     xA = pshot_360.predict_proba(pass_X)[:,1] * xg_model.predict_proba(shot_X)[:,1]")
print("non-key pass: xA = pshot_360.predict_proba(pass_X)[:,1] * zone_xg_lookup(end_x, end_y)")
